##Climate Trace Dataset Wrangling, Joining and Unions

Source: https://huggingface.co/datasets/tjhunter/climate-trace/tree/main/v3-2024-ct5

In [391]:
# Dependencies

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, when, max, min, count
from pyspark.sql.types import StringType
from pyspark.sql.functions import countDistinct
from IPython.display import display

# Create a SparkSession
spark = SparkSession.builder.appName("ClimateTrace").getOrCreate()

In [392]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [393]:
# Parquet Files
parquet_files = {
    "2021_ch4.parquet" : 2021,
    "2021_co2.parquet" : 2021,
    "2021_n2o.parquet" : 2021,
    "2021_co2e_100yr.parquet" : 2021,
    "2022_ch4.parquet" : 2022,
    "2022_co2.parquet" : 2022,
    "2022_n2o.parquet" : 2022,
    "2022_co2e_100yr.parquet" : 2022,
    "2023_ch4.parquet" : 2023,
    "2023_co2.parquet" : 2023,
    "2023_n2o.parquet" : 2023,
    "2023_co2e_100yr.parquet" : 2023,
    "2024_ch4.parquet" : 2024,
    "2024_co2.parquet" : 2024,
    "2024_n2o.parquet" : 2024,
    "2024_co2e_100yr.parquet" : 2024
}
print("Parquet Files in the directory:\n")
for each in parquet_files:
    print(each)

target_parquet = input("Enter the target parquet file name to upload: ").strip()
df = None

if target_parquet in parquet_files:
    parquet_file = f"/content/drive/MyDrive/School Projects/Climate Trace Analysis/raw_data/{parquet_files[target_parquet]}/{target_parquet}"
    df = spark.read.parquet(parquet_file)
    print("Parquet file loaded successfully.")
else:
    print("Invalid input. Please enter a valid parquet file name.")

Parquet Files in the directory:

2021_ch4.parquet
2021_co2.parquet
2021_n2o.parquet
2021_co2e_100yr.parquet
2022_ch4.parquet
2022_co2.parquet
2022_n2o.parquet
2022_co2e_100yr.parquet
2023_ch4.parquet
2023_co2.parquet
2023_n2o.parquet
2023_co2e_100yr.parquet
2024_ch4.parquet
2024_co2.parquet
2024_n2o.parquet
2024_co2e_100yr.parquet
Enter the target parquet file name to upload: 2021_co2.parquet
Parquet file loaded successfully.


In [394]:
# Structure Overview
print("Structure Overview:")
df.printSchema()

Structure Overview:
root
 |-- source_id: decimal(20,0) (nullable = true)
 |-- iso3_country: string (nullable = true)
 |-- sector: string (nullable = true)
 |-- subsector: string (nullable = true)
 |-- original_inventory_sector: string (nullable = true)
 |-- start_time: timestamp (nullable = true)
 |-- end_time: timestamp (nullable = true)
 |-- temporal_granularity: string (nullable = true)
 |-- gas: string (nullable = true)
 |-- emissions_quantity: double (nullable = true)
 |-- emissions_factor: double (nullable = true)
 |-- emissions_factor_units: string (nullable = true)
 |-- capacity: double (nullable = true)
 |-- capacity_units: string (nullable = true)
 |-- capacity_factor: double (nullable = true)
 |-- activity: double (nullable = true)
 |-- activity_units: string (nullable = true)
 |-- created_date: timestamp (nullable = true)
 |-- modified_date: timestamp (nullable = true)
 |-- source_name: string (nullable = true)
 |-- source_type: string (nullable = true)
 |-- lat: double (nu

## <span style="color: white; font-weight: bold; text-decoration: underline;">Feature Selection</span>
Drop the unnecessary and noisy fields to avoid overfitting while lightening the data load.

Actions:
1. Drop the columns
  
  a. other1-other12 (inconsistent and contains missing values that cannot be imputed)

  b. other1_def- other12_def (unnecessary due to the dropping of the other1-other12 fields)

  c. measurement unit labels (redundant)

  d. original inventory sector (has no row content)

  e. modified data (has no row content)


In [395]:
df = df.drop(
    "other1", "other2", "other3",
    "other4", "other5", "other6",
    "other7", "other8", "other9",
    "other10", "other11", "other12",
    "other1_def", "other2_def", "other3_def",
    "other4_def", "other5_def", "other6_def",
    "other7_def", "other8_def", "other9_def",
    "other10_def", "other11_def", "other12_def",
    "emissions_factor_units", "capacity_units", "activity_units",
    "original_inventory_sector", "temporal_granularity", "modified_date"
)

print("Successfully dropped unnecessary fields.")
df.printSchema()

Successfully dropped unnecessary fields.
root
 |-- source_id: decimal(20,0) (nullable = true)
 |-- iso3_country: string (nullable = true)
 |-- sector: string (nullable = true)
 |-- subsector: string (nullable = true)
 |-- start_time: timestamp (nullable = true)
 |-- end_time: timestamp (nullable = true)
 |-- gas: string (nullable = true)
 |-- emissions_quantity: double (nullable = true)
 |-- emissions_factor: double (nullable = true)
 |-- capacity: double (nullable = true)
 |-- capacity_factor: double (nullable = true)
 |-- activity: double (nullable = true)
 |-- created_date: timestamp (nullable = true)
 |-- source_name: string (nullable = true)
 |-- source_type: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- geometry_ref: string (nullable = true)
 |-- conf_source_type: string (nullable = true)
 |-- conf_capacity: string (nullable = true)
 |-- conf_capacity_factor: string (nullable = true)
 |-- conf_activity: string (nullable = true

## <span style="color: white; font-weight: bold; text-decoration: underline;">Data Wrangling</span>
Fill or drop the fields with missing values, rename columns, sanitize to standard versions, normalize if possible.

In [396]:
# iso3_country wrangling
"""
    There are 2 countries that are not in the ISO 3166-1 alpha-3 standard:
    - UNK
    - ZNC

    UNK is unknown and ZNC is a country code in Africa for unlisted countries.
    Replace ZNC with UNK.
"""

df = df.withColumn("iso3_country", when(col("iso3_country") == "ZNC", "UNK").otherwise(col("iso3_country")))

# Rename the field to country_code
df = df.withColumnRenamed("iso3_country", "country_code")

# Validate
df.select("country_code").distinct().show()

print("Null Count:")
print(df.filter(col("country_code").isNull()).count())
print("\n")
df.printSchema()

+------------+
|country_code|
+------------+
|         NIU|
|         CCK|
|         HTI|
|         PSE|
|         POL|
|         BRB|
|         LVA|
|         JAM|
|         ZMB|
|         BRA|
|         SPM|
|         ARM|
|         MOZ|
|         JOR|
|         CUB|
|         FRA|
|         SOM|
|         ABW|
|         TCA|
|         COD|
+------------+
only showing top 20 rows

Null Count:
0


root
 |-- source_id: decimal(20,0) (nullable = true)
 |-- country_code: string (nullable = true)
 |-- sector: string (nullable = true)
 |-- subsector: string (nullable = true)
 |-- start_time: timestamp (nullable = true)
 |-- end_time: timestamp (nullable = true)
 |-- gas: string (nullable = true)
 |-- emissions_quantity: double (nullable = true)
 |-- emissions_factor: double (nullable = true)
 |-- capacity: double (nullable = true)
 |-- capacity_factor: double (nullable = true)
 |-- activity: double (nullable = true)
 |-- created_date: timestamp (nullable = true)
 |-- source_name: string (n

In [397]:
# source id wrangling
df = df.withColumn("source_id", col("source_id").cast("int"))

# Validate
df.select("source_id").distinct().show()

df.agg(
    max(col("emissions_factor")).alias("Max Value"),
    min(col("emissions_factor")).alias("Min Value")
).show()

print("Null Count:")
print(df.filter(col("source_id").isNull()).count())
print("\n")
df.printSchema()

+---------+
|source_id|
+---------+
|  3672954|
|  3673031|
|  3672999|
|  3674889|
|  3674890|
|  1896050|
| 32438322|
|  1895913|
| 32437455|
| 32438817|
| 32438804|
| 32437216|
| 32437565|
|  1896748|
|  1897562|
| 33999133|
| 33999167|
| 33998924|
| 33999016|
|    15447|
+---------+
only showing top 20 rows

+-----------------+-------------------+
|        Max Value|          Min Value|
+-----------------+-------------------+
|5039858.208500672|-27.906776251741505|
+-----------------+-------------------+

Null Count:
0


root
 |-- source_id: integer (nullable = true)
 |-- country_code: string (nullable = true)
 |-- sector: string (nullable = true)
 |-- subsector: string (nullable = true)
 |-- start_time: timestamp (nullable = true)
 |-- end_time: timestamp (nullable = true)
 |-- gas: string (nullable = true)
 |-- emissions_quantity: double (nullable = true)
 |-- emissions_factor: double (nullable = true)
 |-- capacity: double (nullable = true)
 |-- capacity_factor: double (nullable

In [398]:
# sector wrangling

from pyspark.sql.functions import regexp_replace, trim

# replace dashes with space to avoid machine misinterpretation
# trim the values to remove trailing spaces
df = df.withColumn("sector", trim(regexp_replace("sector", "-", " ")))

# Validate
df.select("sector").distinct().show()

print("Null Count:")
print(df.filter(col("sector").isNull()).count())
print("\n")
df.printSchema()


+--------------------+
|              sector|
+--------------------+
|           buildings|
|               power|
|         agriculture|
|  mineral extraction|
|               waste|
|       manufacturing|
|fossil fuel opera...|
|forestry and land...|
|      transportation|
+--------------------+

Null Count:
0


root
 |-- source_id: integer (nullable = true)
 |-- country_code: string (nullable = true)
 |-- sector: string (nullable = true)
 |-- subsector: string (nullable = true)
 |-- start_time: timestamp (nullable = true)
 |-- end_time: timestamp (nullable = true)
 |-- gas: string (nullable = true)
 |-- emissions_quantity: double (nullable = true)
 |-- emissions_factor: double (nullable = true)
 |-- capacity: double (nullable = true)
 |-- capacity_factor: double (nullable = true)
 |-- activity: double (nullable = true)
 |-- created_date: timestamp (nullable = true)
 |-- source_name: string (nullable = true)
 |-- source_type: string (nullable = true)
 |-- lat: double (nullable = true

In [399]:
# subsector wrangling

from pyspark.sql.functions import regexp_replace, trim

# replace dashes with space to avoid machine misinterpretation
# trim the values to remove trailing spaces
df = df.withColumn("subsector", trim(regexp_replace("subsector", "-", " ")))

# Validate
df.select("subsector").distinct().show()

print("Null Count:")
print(df.filter(col("subsector").isNull()).count())
print("\n")
df.printSchema()


+--------------------+
|           subsector|
+--------------------+
|   domestic shipping|
|            aluminum|
|      bauxite mining|
|electricity gener...|
|international shi...|
|enteric fermentat...|
|      iron and steel|
|   domestic aviation|
|      net shrubgrass|
|       copper mining|
|international avi...|
|         iron mining|
|food beverage tob...|
|forest land degra...|
|               glass|
|         net wetland|
|           chemicals|
|      cropland fires|
|domestic wastewat...|
|     net forest land|
+--------------------+
only showing top 20 rows

Null Count:
0


root
 |-- source_id: integer (nullable = true)
 |-- country_code: string (nullable = true)
 |-- sector: string (nullable = true)
 |-- subsector: string (nullable = true)
 |-- start_time: timestamp (nullable = true)
 |-- end_time: timestamp (nullable = true)
 |-- gas: string (nullable = true)
 |-- emissions_quantity: double (nullable = true)
 |-- emissions_factor: double (nullable = true)
 |-- capacity: 

In [400]:
# gas wrangling

# rename gas to gas_type
df = df.withColumnRenamed("gas", "gas_type")

# Validate
df.select("gas_type").distinct().show()

print("Null Count:")
print(df.filter(col("gas_type").isNull()).count())
print("\n")
df.printSchema()

+--------+
|gas_type|
+--------+
|     co2|
+--------+

Null Count:
0


root
 |-- source_id: integer (nullable = true)
 |-- country_code: string (nullable = true)
 |-- sector: string (nullable = true)
 |-- subsector: string (nullable = true)
 |-- start_time: timestamp (nullable = true)
 |-- end_time: timestamp (nullable = true)
 |-- gas_type: string (nullable = true)
 |-- emissions_quantity: double (nullable = true)
 |-- emissions_factor: double (nullable = true)
 |-- capacity: double (nullable = true)
 |-- capacity_factor: double (nullable = true)
 |-- activity: double (nullable = true)
 |-- created_date: timestamp (nullable = true)
 |-- source_name: string (nullable = true)
 |-- source_type: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- geometry_ref: string (nullable = true)
 |-- conf_source_type: string (nullable = true)
 |-- conf_capacity: string (nullable = true)
 |-- conf_capacity_factor: string (nullable = true)
 |-- conf_act

In [401]:
# start_time wrangling

# cast to datetype
df = df.withColumn("start_time", col("start_time").cast("date"))

# rename to start_date
df = df.withColumnRenamed("start_time", "start_date")

# Validate
df.select("start_date").distinct().show()

print("Null Count:")
print(df.filter(col("start_date").isNull()).count())
print("\n")
df.printSchema()

+----------+
|start_date|
+----------+
|2021-11-01|
|2021-06-01|
|2021-08-01|
|2021-12-01|
|2021-02-01|
|2021-07-01|
|2021-01-01|
|2021-03-01|
|2021-10-01|
|2021-09-01|
|2021-04-01|
|2021-05-01|
+----------+

Null Count:
0


root
 |-- source_id: integer (nullable = true)
 |-- country_code: string (nullable = true)
 |-- sector: string (nullable = true)
 |-- subsector: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_time: timestamp (nullable = true)
 |-- gas_type: string (nullable = true)
 |-- emissions_quantity: double (nullable = true)
 |-- emissions_factor: double (nullable = true)
 |-- capacity: double (nullable = true)
 |-- capacity_factor: double (nullable = true)
 |-- activity: double (nullable = true)
 |-- created_date: timestamp (nullable = true)
 |-- source_name: string (nullable = true)
 |-- source_type: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- geometry_ref: string (nullable = true)
 |-- conf_

In [402]:
# end_time wrangling

# cast to datetype
df = df.withColumn("end_time", col("end_time").cast("date"))

# rename to start_date
df = df.withColumnRenamed("end_time", "end_date")

# Validate
df.select("end_date").distinct().show()

print("Null Count:")
print(df.filter(col("end_date").isNull()).count())
print("\n")
df.printSchema()

+----------+
|  end_date|
+----------+
|2021-09-30|
|2021-05-31|
|2021-07-31|
|2021-04-30|
|2021-10-31|
|2021-01-31|
|2021-06-30|
|2021-12-31|
|2021-08-31|
|2021-11-30|
|2021-03-31|
|2021-02-28|
+----------+

Null Count:
0


root
 |-- source_id: integer (nullable = true)
 |-- country_code: string (nullable = true)
 |-- sector: string (nullable = true)
 |-- subsector: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- gas_type: string (nullable = true)
 |-- emissions_quantity: double (nullable = true)
 |-- emissions_factor: double (nullable = true)
 |-- capacity: double (nullable = true)
 |-- capacity_factor: double (nullable = true)
 |-- activity: double (nullable = true)
 |-- created_date: timestamp (nullable = true)
 |-- source_name: string (nullable = true)
 |-- source_type: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- geometry_ref: string (nullable = true)
 |-- conf_sourc

In [403]:
# emissions_factor wrangling

# replace null values with the min value zero
df = df.fillna(0, subset=["emissions_factor"])
df.select("emissions_factor").distinct().show()

df.agg(
    max(col("emissions_factor")).alias("Max Value"),
    min(col("emissions_factor")).alias("Min Value")
).show()

# rename to emissions_factor_ton
df = df.withColumnRenamed("emissions_factor", "emissions_factor_ton")

# Validate
print("Null Count:")
print(df.filter(col("emissions_factor_ton").isNull()).count())
print("\n")
df.printSchema()

+--------------------+
|    emissions_factor|
+--------------------+
|               2.952|
|  2.9519999999999995|
|9.082737637785423E-5|
|    1.54436414879E-4|
|   56.97468972601644|
|  109.73133230655657|
|   2.609200905962E-4|
|  0.0013097102750243|
|  0.3127836837337275|
|   3.393755731062E-4|
|   39.92700675851449|
|  0.6917185353195758|
|   2.638387640129018|
|  1.3474422738469087|
|   4.388551964395536|
|  0.0076129298489398|
|   31.25450846443735|
|  0.0088104127610758|
|   1.617267789892861|
|   0.863685057905845|
+--------------------+
only showing top 20 rows

+-----------------+-------------------+
|        Max Value|          Min Value|
+-----------------+-------------------+
|5039858.208500672|-27.906776251741505|
+-----------------+-------------------+

Null Count:
0


root
 |-- source_id: integer (nullable = true)
 |-- country_code: string (nullable = true)
 |-- sector: string (nullable = true)
 |-- subsector: string (nullable = true)
 |-- start_date: date (nullable = t

In [404]:
# capacity wrangling

# replace inf with NaN
df = df.withColumn("capacity", when(col("capacity") == float("inf"), None).otherwise(col("capacity")))

# rename to capacity_ha
df = df.withColumnRenamed("capacity", "capacity_ha")

# Validate
df.select("capacity_ha").distinct().show()
df.agg(
    max(col("capacity_ha")).alias("Max Value"),
    min(col("capacity_ha")).alias("Min Value")
).show()

print("Null Count:")
print(df.filter(col("capacity_ha").isNull()).count())
print("\n")
df.printSchema()

# the null values will be handled the data scientists in machine learning
# the values cannot be imputed without proper understanding of the implications
# Furthermore, every instance is relevant for a potential time series analysis and dropping one row might largely affect the flow
# NOTE: Missing Not at Random


+--------------------+
|         capacity_ha|
+--------------------+
|  61830.136986301375|
|   7542474.304542382|
|   3069183.865744395|
|   4886630.671018179|
|   2883810.185996585|
|    5266236.42158513|
|  198962.37500790076|
| 4.524830225184558E8|
|   515258.9588270609|
|     274375.39557541|
|  270381.16918412683|
|   2289421.902385844|
|  2031656.0551028333|
|  1243838.6110961435|
|   507257.1977591293|
|   202982.9096122922|
| 2.539182667810216E8|
|   7276535.678516726|
|1.3602078665792936E7|
|    12132.7986976261|
+--------------------+
only showing top 20 rows

+-------------------+------------------+
|          Max Value|         Min Value|
+-------------------+------------------+
|7.570077370388509E9|-7.388508129416005|
+-------------------+------------------+

Null Count:
6948


root
 |-- source_id: integer (nullable = true)
 |-- country_code: string (nullable = true)
 |-- sector: string (nullable = true)
 |-- subsector: string (nullable = true)
 |-- start_date: date (null

In [405]:
# capacity factor wrangling

# fill null with zero
df = df.fillna(0, subset=["capacity_factor"])

# Validate
df.select("capacity_factor").distinct().show()
df.agg(
    max(col("capacity_factor")).alias("Max Value"),
    min(col("capacity_factor")).alias("Min Value")
).show()

print("Null Count:")
print(df.filter(col("capacity_factor").isNull()).count())
print("\n")
df.printSchema()


+------------------+
|   capacity_factor|
+------------------+
|0.8575212832036246|
|0.0406823255813953|
|0.3960783333333333|
|0.8752142857142857|
|0.0273271428571428|
|0.5101571428571429|
|0.0257066666666666|
| 0.377961394302396|
|0.0433518518518518|
|        0.91893125|
|0.6510707692307692|
|0.8322928571428571|
|        0.36590875|
|          0.898625|
|0.8903264705882351|
|0.0389174999999999|
|0.8157941379310345|
|           0.22902|
|0.8157590625000001|
|0.3501758333333333|
+------------------+
only showing top 20 rows

+--------------------+---------+
|           Max Value|Min Value|
+--------------------+---------+
|1.2577829880592796E8|      0.0|
+--------------------+---------+

Null Count:
0


root
 |-- source_id: integer (nullable = true)
 |-- country_code: string (nullable = true)
 |-- sector: string (nullable = true)
 |-- subsector: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- gas_type: string (nullable = true)


In [406]:
# activity wrangling

# rename to activity_ha
df = df.withColumnRenamed("activity", "activity_ha")

# Validate
df.select("activity_ha").distinct().show()
df.agg(
    max(col("activity_ha")).alias("Max Value"),
    min(col("activity_ha")).alias("Min Value")
).show()

print("Null Count:")
print(df.filter(col("activity_ha").isNull()).count())
print("\n")
df.printSchema()

# The null values will be handled by the data science team prior to machine learning
# The null values cannot be imputed with random values without fully understanding the implications
# Furthermore, every instance is relevant for a potential time series analysis and dropping one row might largely affect the flow
# NOTE: Missing Not at Random


+------------------+
|       activity_ha|
+------------------+
|  195610.337972167|
|27425.978296612957|
|29988.819467280497|
| 55676.99221747231|
| 4961.966856832382|
|  14349.3615865254|
| 21800.64308681672|
|23562.643867148963|
| 39985.09262304066|
|25652.428320655352|
|1386.2856082397698|
| 21204.74158413709|
| 186115.3096187401|
|        55438.8109|
|29311.311325714287|
|21287.769507142857|
|32443.873181818177|
| 17029.07701935484|
|12258.737626451612|
| 75696.00697681455|
+------------------+
only showing top 20 rows

+--------------------+------------------+
|           Max Value|         Min Value|
+--------------------+------------------+
|7.462960084532733E11|-40.04431996056479|
+--------------------+------------------+

Null Count:
6864


root
 |-- source_id: integer (nullable = true)
 |-- country_code: string (nullable = true)
 |-- sector: string (nullable = true)
 |-- subsector: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable 

In [407]:
# created_date wrangling

# cast from timestamp to date
df = df.withColumn("created_date", col("created_date").cast("date"))

# Validate
df.select("created_date").distinct().show()
df.agg(
    max(col("created_date")).alias("Max Value"),
    min(col("created_date")).alias("Min Value")
).show()

print("Null Count:")
print(df.filter(col("created_date").isNull()).count())
print("\n")
df.printSchema()

+------------+
|created_date|
+------------+
|  2024-09-18|
|  2023-09-14|
|  2024-10-02|
|  2024-01-11|
|  2024-08-04|
|  2024-08-09|
|  2024-08-03|
|  2023-10-06|
|  2024-09-01|
|  2024-09-25|
|  2024-10-01|
|  2024-10-04|
|  2023-09-15|
|  2024-09-19|
|  2024-10-15|
|  2024-09-09|
|  2024-07-03|
|  2023-10-18|
|  2024-08-02|
|  2024-10-07|
+------------+
only showing top 20 rows

+----------+----------+
| Max Value| Min Value|
+----------+----------+
|2024-10-18|2023-09-07|
+----------+----------+

Null Count:
0


root
 |-- source_id: integer (nullable = true)
 |-- country_code: string (nullable = true)
 |-- sector: string (nullable = true)
 |-- subsector: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- gas_type: string (nullable = true)
 |-- emissions_quantity: double (nullable = true)
 |-- emissions_factor_ton: double (nullable = false)
 |-- capacity_ha: double (nullable = true)
 |-- capacity_factor: double (nullable = fa

In [408]:
# source name wrangling

# due to irrelevance and data inconsistencies, this should be dropped
df = df.drop("source_name")

# Validate
df.printSchema()

root
 |-- source_id: integer (nullable = true)
 |-- country_code: string (nullable = true)
 |-- sector: string (nullable = true)
 |-- subsector: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- gas_type: string (nullable = true)
 |-- emissions_quantity: double (nullable = true)
 |-- emissions_factor_ton: double (nullable = false)
 |-- capacity_ha: double (nullable = true)
 |-- capacity_factor: double (nullable = false)
 |-- activity_ha: double (nullable = true)
 |-- created_date: date (nullable = true)
 |-- source_type: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- geometry_ref: string (nullable = true)
 |-- conf_source_type: string (nullable = true)
 |-- conf_capacity: string (nullable = true)
 |-- conf_capacity_factor: string (nullable = true)
 |-- conf_activity: string (nullable = true)
 |-- conf_emissions_factor: string (nullable = true)
 |-- conf_emissions_quantity: str

In [409]:
# source_type wrangling

# handle null values
df = df.withColumn("source_type", when(col("source_type").isNull(), "unknown").otherwise(col("source_type")))

# Validate
df.select("source_type").distinct().show()

print("Null Count:")
print(df.filter(col("source_type").isNull()).count())
print("\n")
df.printSchema()

+--------------------+
|         source_type|
+--------------------+
|      DRI-EAF,BF/BOF|
|manufacturer | Li...|
|Certification - E...|
|Meat Processing| ...|
|coal, other_fossi...|
|Certification - E...|
|       biomass, coal|
|Meat Processing| ...|
|Imported Product|...|
|Identification - ...|
|manufacturer | Li...|
|manufacturer | Li...|
|Certification - E...|
|Identification - ...|
|Certification - E...|
|Meat Processing| ...|
|Meat Processing| ...|
|enteric_fermentat...|
|Meat Processing| ...|
|Meat Processing| ...|
+--------------------+
only showing top 20 rows

Null Count:
0


root
 |-- source_id: integer (nullable = true)
 |-- country_code: string (nullable = true)
 |-- sector: string (nullable = true)
 |-- subsector: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- gas_type: string (nullable = true)
 |-- emissions_quantity: double (nullable = true)
 |-- emissions_factor_ton: double (nullable = false)
 |-- capacity_h

In [410]:
# lat and lon data wrangling

# load the supplementary dataset
parquet_file = f"/content/drive/MyDrive/School Projects/Climate Trace Analysis/raw_data/geo/climate_trace_points_v3_2024_ct5.parquet"
geo_points_df_main = spark.read.parquet(parquet_file)
print("Geo points parquet file loaded successfully.")
geo_points_df_main.printSchema()

# select the necessary fields
geo_points_df = geo_points_df_main.select("geometry_ref", "lat", "lng")
geo_points_df = geo_points_df.withColumnRenamed("lng", "lon_complete")
geo_points_df = geo_points_df.withColumnRenamed("lat", "lat_complete")
geo_points_df.printSchema()

# map the geometry_ref from the df to the lat and lng values of geo_points_df
df = df.join(geo_points_df, on="geometry_ref", how="left")

# Validate
print("Null Count:")
print(df.filter(col("lat_complete").isNull() | col("lon_complete").isNull()).count())
print("\n")
df.printSchema()
df.select("geometry_ref","lat").where(col("lat").isNull()).show(10)

# The geometry ref of the current raw dataset with null coordinates do not have equivalent trace points in the
# related trace_points dataset.

# The alternative solution is to join it using the iso3_country and country_code as these
# are the common fields between the two datasets


Geo points parquet file loaded successfully.
root
 |-- geometry_ref: string (nullable = true)
 |-- gadm: string (nullable = true)
 |-- geom_wkb: binary (nullable = true)
 |-- lat: double (nullable = true)
 |-- lng: double (nullable = true)
 |-- gadm_0: string (nullable = true)
 |-- gadm_1: string (nullable = true)
 |-- gadm_2: string (nullable = true)
 |-- gadm_level: long (nullable = true)
 |-- iso3_country: string (nullable = true)

root
 |-- geometry_ref: string (nullable = true)
 |-- lat_complete: double (nullable = true)
 |-- lon_complete: double (nullable = true)

Null Count:
11548968


root
 |-- geometry_ref: string (nullable = true)
 |-- source_id: integer (nullable = true)
 |-- country_code: string (nullable = true)
 |-- sector: string (nullable = true)
 |-- subsector: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- gas_type: string (nullable = true)
 |-- emissions_quantity: double (nullable = true)
 |-- emissions_fac

In [411]:
# alternative join method to fill null

# drop the lat_complete and lon_complete
df = df.drop("lat_complete", "lon_complete")

geo_points_df = geo_points_df_main.select("iso3_country", "lat", "lng")
geo_points_df = geo_points_df.withColumnRenamed("lng", "lon_complete")
geo_points_df = geo_points_df.withColumnRenamed("lat", "lat_complete")
geo_points_df.printSchema()

# drop duplicate iso3_country
geo_points_df = geo_points_df.dropDuplicates(["iso3_country"])

# map the gadm from the df to the lat and lng values of geo_points_df
df = df.join(geo_points_df, df["country_code"] == geo_points_df["iso3_country"], "left")

# drop the incomplete lon and lat fields
df = df.drop("lon", "lat")

# drop geometry_ref
df = df.drop("geometry_ref")

# drop iso3_country
df = df.drop("iso3_country")

# rename the coordinates to a more descriptive name
df = df.withColumnRenamed("lon_complete", "longitude")
df = df.withColumnRenamed("lat_complete", "latitude")
# Validate
print("Null Count:")
print(df.filter(col("lat_complete").isNull() | col("lon_complete").isNull()).count())
print("\n")
df.printSchema()

# The null fill is successful. The null instances are due to the unknown country of origin
df.select("latitude", "longitude" , "country_code").where(col("lat_complete").isNull() | col("lon_complete").isNull()).show(10)

root
 |-- iso3_country: string (nullable = true)
 |-- lat_complete: double (nullable = true)
 |-- lon_complete: double (nullable = true)

Null Count:
5604


root
 |-- source_id: integer (nullable = true)
 |-- country_code: string (nullable = true)
 |-- sector: string (nullable = true)
 |-- subsector: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- gas_type: string (nullable = true)
 |-- emissions_quantity: double (nullable = true)
 |-- emissions_factor_ton: double (nullable = false)
 |-- capacity_ha: double (nullable = true)
 |-- capacity_factor: double (nullable = false)
 |-- activity_ha: double (nullable = true)
 |-- created_date: date (nullable = true)
 |-- source_type: string (nullable = true)
 |-- conf_source_type: string (nullable = true)
 |-- conf_capacity: string (nullable = true)
 |-- conf_capacity_factor: string (nullable = true)
 |-- conf_activity: string (nullable = true)
 |-- conf_emissions_factor: string (nullabl

In [412]:
# year wrangling

# cast to int
df = df.withColumn("year", col("year").cast("int"))

# Validate
print("Distinct Values:")
df.select("year").distinct().show()
print("Null Count:")
print(df.filter(col("year").isNull()).count())
print("\n")
df.printSchema()

Distinct Values:
+----+
|year|
+----+
|2021|
+----+

Null Count:
0


root
 |-- source_id: integer (nullable = true)
 |-- country_code: string (nullable = true)
 |-- sector: string (nullable = true)
 |-- subsector: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- gas_type: string (nullable = true)
 |-- emissions_quantity: double (nullable = true)
 |-- emissions_factor_ton: double (nullable = false)
 |-- capacity_ha: double (nullable = true)
 |-- capacity_factor: double (nullable = false)
 |-- activity_ha: double (nullable = true)
 |-- created_date: date (nullable = true)
 |-- source_type: string (nullable = true)
 |-- conf_source_type: string (nullable = true)
 |-- conf_capacity: string (nullable = true)
 |-- conf_capacity_factor: string (nullable = true)
 |-- conf_activity: string (nullable = true)
 |-- conf_emissions_factor: string (nullable = true)
 |-- conf_emissions_quantity: string (nullable = true)
 |-- year: integer (nul

In [413]:
# confidence metrics wrangling

# rename to more descriptive labels
df = df.withColumnRenamed("conf_source_type", "confidence_source_type")
df = df.withColumnRenamed("conf_capacity", "confidence_capacity")
df = df.withColumnRenamed("conf_capacity_factor", "confidence_capacity_factor")
df = df.withColumnRenamed("conf_activity", "confidence_activity")
df = df.withColumnRenamed("conf_emissions_factor", "confidence_emissions_factor")
df = df.withColumnRenamed("conf_emissions_quantity", "confidence_emissions_quantity")

# Validate
df.printSchema()

root
 |-- source_id: integer (nullable = true)
 |-- country_code: string (nullable = true)
 |-- sector: string (nullable = true)
 |-- subsector: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- gas_type: string (nullable = true)
 |-- emissions_quantity: double (nullable = true)
 |-- emissions_factor_ton: double (nullable = false)
 |-- capacity_ha: double (nullable = true)
 |-- capacity_factor: double (nullable = false)
 |-- activity_ha: double (nullable = true)
 |-- created_date: date (nullable = true)
 |-- source_type: string (nullable = true)
 |-- confidence_source_type: string (nullable = true)
 |-- confidence_capacity: string (nullable = true)
 |-- confidence_capacity_factor: string (nullable = true)
 |-- confidence_activity: string (nullable = true)
 |-- confidence_emissions_factor: string (nullable = true)
 |-- confidence_emissions_quantity: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- latitude: doub

## <span style="color: white; font-weight: bold; text-decoration: underline;">Feature Engineering</span>
Create derived fields from the existing dataset to provide more insight.

Actions:
1. Add full country name field.

2. Add month and quarter fields for expanded analysis.

In [414]:
!pip install country_converter

In [415]:
import country_converter as coco

# load coco
cc = coco.CountryConverter()

# map country code to full country name
unique_codes = df.select("country_code").distinct()
country_data = [(code, cc.convert(names=code, to="name")) for code in df.select("country_code").distinct().rdd.flatMap(lambda x: x).collect()]

# Convert dictionary to a PySpark DataFrame
country_map_df = spark.createDataFrame(country_data, ["country_code", "country_name"])

# Join with the original DataFrame
df = df.join(country_map_df, "country_code", "left")

# fill null with unknown
df = df.replace("not found", "unknown", subset=["country_name"])

# Validation
df.select("country_name").show(10)

print("Distinct Values:")
df.select("country_name").distinct().show()
print("Null Count:")
print(df.filter(col("country_name").isNull()).count())
print("\n")
df.printSchema()


+--------------------+
|        country_name|
+--------------------+
|United Arab Emirates|
|United Arab Emirates|
|United Arab Emirates|
|United Arab Emirates|
|United Arab Emirates|
|United Arab Emirates|
|United Arab Emirates|
|United Arab Emirates|
|United Arab Emirates|
|United Arab Emirates|
+--------------------+
only showing top 10 rows

Distinct Values:
+--------------------+
|        country_name|
+--------------------+
|                Chad|
|              Russia|
|            Paraguay|
|            Anguilla|
|               Yemen|
|British Indian Oc...|
|             Senegal|
|South Georgia and...|
|              Sweden|
|          Cabo Verde|
|             Tokelau|
|            Kiribati|
|French Southern T...|
|              Guyana|
|             Eritrea|
|         Philippines|
|              Jersey|
|            Djibouti|
|               Tonga|
|      Norfolk Island|
+--------------------+
only showing top 20 rows

Null Count:
0


root
 |-- country_code: string (nullable 

In [416]:
# add month field
from pyspark.sql.functions import month

df = df.withColumn("month", month(df["start_date"]))

# validate month
df.select("month").show(10)

print("Distinct Values:")
df.select("month").distinct().show()
print("Null Count:")
print(df.filter(col("month").isNull()).count())

# add quarter field
from pyspark.sql.functions import quarter

df = df.withColumn("quarter", quarter(df["start_date"]))

# validate quarter
df.select("quarter").show(10)

print("Distinct Values:")
df.select("quarter").distinct().show()
print("Null Count:")
print(df.filter(col("quarter").isNull()).count())
print("\n")
df.printSchema()

+-----+
|month|
+-----+
|    1|
|    2|
|    3|
|    4|
|    5|
|    6|
|    7|
|    8|
|    9|
|   10|
+-----+
only showing top 10 rows

Distinct Values:
+-----+
|month|
+-----+
|   12|
|    1|
|    6|
|    3|
|    5|
|    9|
|    4|
|    8|
|    7|
|   10|
|   11|
|    2|
+-----+

Null Count:
0
+-------+
|quarter|
+-------+
|      1|
|      1|
|      1|
|      2|
|      2|
|      2|
|      3|
|      3|
|      3|
|      4|
+-------+
only showing top 10 rows

Distinct Values:
+-------+
|quarter|
+-------+
|      1|
|      3|
|      4|
|      2|
+-------+

Null Count:
0


root
 |-- country_code: string (nullable = true)
 |-- source_id: integer (nullable = true)
 |-- sector: string (nullable = true)
 |-- subsector: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- gas_type: string (nullable = true)
 |-- emissions_quantity: double (nullable = true)
 |-- emissions_factor_ton: double (nullable = false)
 |-- capacity_ha: double (nulla

## <span style="color: white; font-weight: bold; text-decoration: underline;">Export Preparations</span>
Prepare the dataset for final store.

Actions:
1. Cast the fields to the most efficient data type.

2. Logically sort the fields.

In [417]:
df.printSchema()

root
 |-- country_code: string (nullable = true)
 |-- source_id: integer (nullable = true)
 |-- sector: string (nullable = true)
 |-- subsector: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- gas_type: string (nullable = true)
 |-- emissions_quantity: double (nullable = true)
 |-- emissions_factor_ton: double (nullable = false)
 |-- capacity_ha: double (nullable = true)
 |-- capacity_factor: double (nullable = false)
 |-- activity_ha: double (nullable = true)
 |-- created_date: date (nullable = true)
 |-- source_type: string (nullable = true)
 |-- confidence_source_type: string (nullable = true)
 |-- confidence_capacity: string (nullable = true)
 |-- confidence_capacity_factor: string (nullable = true)
 |-- confidence_activity: string (nullable = true)
 |-- confidence_emissions_factor: string (nullable = true)
 |-- confidence_emissions_quantity: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- latitude: doub

In [418]:
final_df = df.select(
    "source_id",
    "gas_type",
    "emissions_quantity",
    "emissions_factor_ton",
    "capacity_ha",
    "capacity_factor",
    "activity_ha",
    "sector",
    "subsector",
    "country_code",
    "country_name",
    "year",
    "month",
    "quarter",
    "start_date",
    "end_date",
    "latitude",
    "longitude",
    "created_date",
    "confidence_source_type",
    "confidence_capacity",
    "confidence_capacity_factor",
    "confidence_activity",
    "confidence_emissions_factor",
    "confidence_emissions_quantity"
)

# validate
final_df.printSchema()

root
 |-- source_id: integer (nullable = true)
 |-- gas_type: string (nullable = true)
 |-- emissions_quantity: double (nullable = true)
 |-- emissions_factor_ton: double (nullable = false)
 |-- capacity_ha: double (nullable = true)
 |-- capacity_factor: double (nullable = false)
 |-- activity_ha: double (nullable = true)
 |-- sector: string (nullable = true)
 |-- subsector: string (nullable = true)
 |-- country_code: string (nullable = true)
 |-- country_name: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- created_date: date (nullable = true)
 |-- confidence_source_type: string (nullable = true)
 |-- confidence_capacity: string (nullable = true)
 |-- confidence_capacity_factor: string (nullable = true)
 |-- confidence_activity: stri

## <span style="color: white; font-weight: bold; text-decoration: underline;">Data Export</span>
Export the dataset to parquet for distribution.

In [419]:
# export the dataset to parquet
parquet_file = f"/content/drive/MyDrive/School Projects/Climate Trace Analysis/cleaned_data/{parquet_files[target_parquet]}/cleaned_{target_parquet}"
df.write.mode("overwrite").parquet(parquet_file)
print("Parquet file exported successfully.")

Parquet file exported successfully.


## <span style="color: white; font-weight: bold; text-decoration: underline;">Missing Analysis</span>
Using missingno, show the relationships of missing values.

In [420]:
# import pandas as pd
# import missingno as msno
# import matplotlib.pyplot as plt

# # Convert PySpark DataFrame to Pandas\
# final_df_sample = final_df.limit(10000)
# final_df_sample = final_df_sample.toPandas()

# # Generate missing value matrix
# msno.matrix(final_df_sample)
# plt.show()
# pd_df = final_df.toPandas()

# # Generate missing value matrix
# msno.matrix(pd_df)
# plt.show()